# Validation Deck Selection

Build, inspect, roll, audit, and export the three deterministic isolation-validation selections.
All adjustable parameters are read from `cfg/select_validation_decks.yaml`.

Outputs remain at:

- `data/deck_isolation_selection.csv`
- `data/archetype_isolation_selection.csv`
- `data/top_deck_archetype_isolation_selection.csv`


## Setup and Configuration


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd() / 'imitation_learning', Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'analysis' / 'deck_statistics.py').is_file():
            return candidate.resolve()
    raise FileNotFoundError('Could not locate the imitation_learning project root')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml

from analysis.deck_statistics import (
    CardCatalog,
    annotate_deck_facts,
    build_deck_census,
    load_deck_rows,
)
from analysis.deck_selection import (
    IsolationSamplingError,
    audit_deck_archetype_selection,
    build_archetype_isolation_candidates,
    build_archetype_selection_details,
    build_deck_isolation_candidates,
    build_top_deck_archetype_candidates,
    roll_archetype_isolation_selection,
    roll_deck_isolation_selection,
    roll_top_deck_archetype_selection,
)

CONFIG_PATH = PROJECT_ROOT / 'cfg' / 'select_validation_decks.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    selection_config = yaml.safe_load(handle)

DECK_DATA_DIR = PROJECT_ROOT / selection_config['inputs']['deck_data']
CARD_TABLE_PATH = PROJECT_ROOT / selection_config['inputs']['card_table']
OUTPUT_PATHS = {
    name: PROJECT_ROOT / value
    for name, value in selection_config['outputs'].items()
}
print('Config:', CONFIG_PATH)
print('Deck data:', DECK_DATA_DIR)
print('Card table:', CARD_TABLE_PATH)


## Load & Validate

Every successfully extracted episode must have exactly two rows, players 0 and 1, and each row must contain a valid 60-card list.

In [ ]:
deck_paths = sorted(DECK_DATA_DIR.glob('*.decks.csv'))
if not deck_paths:
    raise FileNotFoundError(f'No .decks.csv files found under {DECK_DATA_DIR}')

deck_facts = load_deck_rows(deck_paths)
catalog = CardCatalog.from_csv(CARD_TABLE_PATH)

print('Files:', len(deck_paths))
print('Deck rows:', len(deck_facts))
print('Episodes:', deck_facts[['date', 'episode_id']].drop_duplicates().shape[0])
print('Dates:', ', '.join(sorted(deck_facts['date'].unique())))

fact_preview = deck_facts.drop(columns=['deck']).head(10)
display(fact_preview)

## Deck Census

A row represents one exact sorted 60-card multiset. Rank follows usage, while `deck_id` remains stable across reruns.

In [ ]:
deck_summary = build_deck_census(deck_facts, catalog)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = OUTPUT_DIR / 'deck_summary.csv'
deck_summary.to_csv(summary_path, index=False, encoding='utf-8-sig')

print('Unique exact decks:', len(deck_summary))
print('Saved:', summary_path)
display(
    deck_summary[
        ['rank', 'deck_archetype', 'deck_id', 'uses', 'usage_percent', 'wins', 'losses', 'draws', 'win_rate', 'unique_opponent_decks']
    ].head(100)
)

## Top-Deck Archetype Isolation Roll

This independent roll selects variants inside the model's final deck archetype. The exact configured top deck is never selected, but it may appear as an opponent in a selected replay. Change `TOP_DECK_ROLL_ID` to reroll.

In [ ]:
top_cfg = selection_config['top_deck_archetype_isolation']
TOP_DECK_ARCHETYPE = top_cfg['archetype']
TOP_DECK_CARD_IDS = top_cfg['card_ids']
TOP_DECK_MIN_REPLAYS = top_cfg['min_validation_replays']
TOP_DECK_MIN_COUNT = top_cfg['min_count']
TOP_DECK_TOTAL_REPLAYS_MIN = top_cfg['total_replays_min']
TOP_DECK_TOTAL_REPLAYS_MAX = top_cfg['total_replays_max']
TOP_DECK_ROLL_ID = top_cfg['roll_id']
TOP_DECK_SELECTION_PATH = OUTPUT_PATHS['top_deck_archetype_isolation']

annotated_facts = annotate_deck_facts(deck_facts, catalog)
top_deck_candidates = build_top_deck_archetype_candidates(
    annotated_facts,
    target_archetype=TOP_DECK_ARCHETYPE,
    top_deck_card_ids=TOP_DECK_CARD_IDS,
    min_validation_replays=TOP_DECK_MIN_REPLAYS,
)
TOP_DECK_ID = top_deck_candidates.attrs['similarity_bands']['top_deck_id']
top_band_info = top_deck_candidates.attrs['similarity_bands']
print('Top deck ID:', TOP_DECK_ID)
print('Target-archetype candidates:', len(top_deck_candidates))
print('Automatic similarity cut points:', top_band_info)
display(
    top_deck_candidates[
        [
            'deck_id', 'deck_archetype', 'replays', 'uses', 'win_rate',
            'top_deck_changed_slots', 'top_deck_weighted_jaccard',
            'similarity_band', 'is_top_deck', 'eligible',
        ]
    ]
)

top_deck_roll = None
selected_top_deck_variants = pd.DataFrame()
try:
    top_deck_roll = roll_top_deck_archetype_selection(
        annotated_facts,
        top_deck_candidates,
        min_count=TOP_DECK_MIN_COUNT,
        total_replays_min=TOP_DECK_TOTAL_REPLAYS_MIN,
        total_replays_max=TOP_DECK_TOTAL_REPLAYS_MAX,
        roll_id=TOP_DECK_ROLL_ID,
    )
    selected_top_deck_variants = top_deck_candidates[
        top_deck_candidates['deck_id'].isin(top_deck_roll.selected)
    ].copy()
    top_selection_order = {
        deck_id: index for index, deck_id in enumerate(top_deck_roll.selected)
    }
    selected_top_deck_variants['selection_order'] = (
        selected_top_deck_variants['deck_id'].map(top_selection_order)
    )
    selected_top_deck_variants['card_ids'] = (
        selected_top_deck_variants['deck_id'].map(
            deck_summary.set_index('deck_id')['deck_card_ids_json']
        )
    )
    selected_top_deck_variants = selected_top_deck_variants.sort_values(
        'selection_order'
    )[
        [
            'deck_archetype', 'deck_id', 'replays', 'uses', 'win_rate',
            'top_deck_changed_slots', 'top_deck_weighted_jaccard',
            'similarity_band', 'card_ids',
        ]
    ]
    display(selected_top_deck_variants)
    print('Selected unique replays:', top_deck_roll.replay_count)
    print('Sampling attempts:', top_deck_roll.attempts)
    selected_top_deck_variants.to_csv(
        TOP_DECK_SELECTION_PATH, index=False, encoding='utf-8-sig'
    )
    print('Saved:', TOP_DECK_SELECTION_PATH)
except (IsolationSamplingError, ValueError) as exc:
    print('Top-deck archetype roll failed:', exc)
    if isinstance(exc, IsolationSamplingError):
        display(pd.DataFrame([exc.diagnostics]))
    print('Existing selection CSV was not overwritten.')

## Deck Isolation Candidates

Filter exact decks by replay support and the Deck-Isolation constraints. Parameters for this selection stay in this section.

In [ ]:
deck_cfg = selection_config['deck_isolation']
DECK_MIN_REPLAYS = deck_cfg['min_validation_replays']

deck_isolation_candidates = build_deck_isolation_candidates(
    annotated_facts,
    min_validation_replays=DECK_MIN_REPLAYS,
    excluded_archetypes=(TOP_DECK_ARCHETYPE,),
)
ordinary_deck_candidates = deck_isolation_candidates[
    deck_isolation_candidates['deck_archetype'].ne(TOP_DECK_ARCHETYPE)
].copy()
eligible_deck_candidates = ordinary_deck_candidates[
    ordinary_deck_candidates['eligible']
].copy()

band_info = deck_isolation_candidates.attrs['similarity_bands']
print('Eligible exact decks:', len(eligible_deck_candidates), '/', len(ordinary_deck_candidates))
print('Automatic similarity cut points:')
display(pd.DataFrame([band_info]))
display(
    ordinary_deck_candidates[
        [
            'deck_id', 'deck_archetype', 'replays', 'uses', 'win_rate',
            'remaining_archetype_decks', 'all_cards_seen_in_train',
            'nearest_train_deck_id', 'nearest_changed_slots',
            'nearest_weighted_jaccard', 'reference_train_deck_id',
            'reference_train_uses', 'reference_changed_slots',
            'reference_weighted_jaccard', 'similarity_band', 'eligible',
        ]
    ]
)

## Deck Isolation Roll

Change `DECK_ROLL_ID` and rerun this cell to obtain another reproducible roll. Selected decks always have different archetypes.

In [ ]:
DECK_MIN_COUNT = deck_cfg['min_count']
DECK_TOTAL_REPLAYS_MIN = deck_cfg['total_replays_min']
DECK_TOTAL_REPLAYS_MAX = deck_cfg['total_replays_max']
DECK_ROLL_ID = deck_cfg['roll_id']
DECK_SELECTION_PATH = OUTPUT_PATHS['deck_isolation']

deck_roll = None
selected_decks = pd.DataFrame()
try:
    deck_roll = roll_deck_isolation_selection(
        annotated_facts,
        deck_isolation_candidates,
        min_count=DECK_MIN_COUNT,
        total_replays_min=DECK_TOTAL_REPLAYS_MIN,
        total_replays_max=DECK_TOTAL_REPLAYS_MAX,
        roll_id=DECK_ROLL_ID,
        excluded_archetypes=(TOP_DECK_ARCHETYPE,),
    )
    selected_decks = deck_isolation_candidates[
        deck_isolation_candidates['deck_id'].isin(deck_roll.selected)
    ].copy()
    selection_order = {deck_id: index for index, deck_id in enumerate(deck_roll.selected)}
    selected_decks['selection_order'] = selected_decks['deck_id'].map(selection_order)
    selected_decks['card_ids'] = selected_decks['deck_id'].map(
        deck_summary.set_index('deck_id')['deck_card_ids_json']
    )
    selected_decks = selected_decks.sort_values('selection_order')
    selected_decks = selected_decks[
        [
            'deck_id', 'deck_archetype', 'replays', 'uses', 'win_rate',
            'similarity_band', 'nearest_train_deck_id',
            'nearest_changed_slots', 'nearest_weighted_jaccard',
            'reference_train_deck_id', 'reference_train_uses',
            'reference_changed_slots', 'reference_weighted_jaccard',
            'card_ids',
        ]
    ]
    display(selected_decks)
    print('Selected unique replays:', deck_roll.replay_count)
    print('Sampling attempts:', deck_roll.attempts)
    selected_decks.to_csv(DECK_SELECTION_PATH, index=False, encoding='utf-8-sig')
    print('Saved:', DECK_SELECTION_PATH)
except (IsolationSamplingError, ValueError) as exc:
    print('Deck Isolation roll failed:', exc)
    if isinstance(exc, IsolationSamplingError):
        display(pd.DataFrame([exc.diagnostics]))
    print('Existing selection CSV was not overwritten.')

In [ ]:
band_colors = {'high': '#D55E00', 'moderate': '#0072B2', 'lower': '#009E73'}
fig, ax = plt.subplots(figsize=(9, 5.5))
for band in ('high', 'moderate', 'lower'):
    group = eligible_deck_candidates[eligible_deck_candidates['similarity_band'] == band]
    if not group.empty:
        ax.scatter(
            group['reference_changed_slots'], group['replays'],
            s=55, alpha=0.8, color=band_colors[band], label=band.title(),
        )
if not selected_decks.empty:
    ax.scatter(
        selected_decks['reference_changed_slots'], selected_decks['replays'],
        s=180, marker='*', color='#F0E442', edgecolor='#222222',
        linewidth=0.8, label='Selected', zorder=5,
    )
ax.set(
    title='Eligible Deck-Isolation Candidates',
    xlabel='Changed Slots to Most-Used Remaining Same-Archetype Deck',
    ylabel='Unique Replays',
)
ax.grid(alpha=0.2)
if not eligible_deck_candidates.empty or not selected_decks.empty:
    ax.legend(title='Similarity')
plt.tight_layout()
plt.show()

## Archetype Isolation Candidates

Explicit rules and curated `named_fallback_main_pokemon` labels are eligible. Raw fallback labels remain visible but are not sampled. Core-card overlap with other labels is descriptive only.

In [ ]:
archetype_cfg = selection_config['archetype_isolation']
ARCHETYPE_MIN_REPLAYS = archetype_cfg['min_validation_replays']

archetype_isolation_candidates = build_archetype_isolation_candidates(
    annotated_facts,
    catalog,
    min_validation_replays=ARCHETYPE_MIN_REPLAYS,
)
ordinary_archetype_candidates = archetype_isolation_candidates[
    archetype_isolation_candidates['deck_archetype'].ne(TOP_DECK_ARCHETYPE)
].copy()
eligible_archetype_candidates = ordinary_archetype_candidates[
    ordinary_archetype_candidates['eligible']
].copy()
print('Eligible curated archetypes:', len(eligible_archetype_candidates), '/', len(ordinary_archetype_candidates))
display(
    ordinary_archetype_candidates[
        [
            'deck_archetype', 'classification_method', 'core_card_names',
            'replays', 'uses', 'win_rate', 'unique_exact_decks',
            'core_card_other_archetype_decks',
            'core_card_other_archetype_replays', 'rule_defined',
            'sampling_method_allowed', 'eligible',
        ]
    ]
)

In [ ]:
archetype_counts = archetype_isolation_candidates.sort_values('unique_exact_decks')
fig_height = max(5, 0.28 * len(archetype_counts))
fig, ax = plt.subplots(figsize=(9, fig_height))
ax.barh(
    archetype_counts['deck_archetype'],
    archetype_counts['unique_exact_decks'],
    color='#4C78A8',
)
ax.set(
    title='Unique Exact Decks per Archetype',
    xlabel='Unique Exact Decks',
    ylabel='Archetype',
)
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
eligible_archetype_plot = eligible_archetype_candidates.sort_values('replays')
fig_height = max(4, 0.35 * len(eligible_archetype_plot))
fig, ax = plt.subplots(figsize=(9, fig_height))
ax.barh(
    eligible_archetype_plot['deck_archetype'],
    eligible_archetype_plot['replays'],
    color='#E45756',
)
ax.axvline(ARCHETYPE_MIN_REPLAYS, color='#333333', linestyle='--', linewidth=1)
ax.set(
    title='Eligible Curated Archetype-Isolation Replay Support',
    xlabel='Unique Replays with This Primary Archetype',
    ylabel='Archetype',
)
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

## Archetype Isolation Roll

Change `ARCHETYPE_ROLL_ID` and rerun this cell to obtain another reproducible roll. Eligible rule or curated named-fallback archetypes have equal sampling weight.

In [ ]:
ARCHETYPE_MIN_COUNT = archetype_cfg['min_count']
ARCHETYPE_TOTAL_REPLAYS_MIN = archetype_cfg['total_replays_min']
ARCHETYPE_TOTAL_REPLAYS_MAX = archetype_cfg['total_replays_max']
ARCHETYPE_ROLL_ID = archetype_cfg['roll_id']
ARCHETYPE_SELECTION_PATH = OUTPUT_PATHS['archetype_isolation']

archetype_roll = None
selected_archetype_summary = pd.DataFrame()
selected_archetype_decks = pd.DataFrame()
try:
    archetype_roll = roll_archetype_isolation_selection(
        annotated_facts,
        archetype_isolation_candidates,
        min_count=ARCHETYPE_MIN_COUNT,
        total_replays_min=ARCHETYPE_TOTAL_REPLAYS_MIN,
        total_replays_max=ARCHETYPE_TOTAL_REPLAYS_MAX,
        roll_id=ARCHETYPE_ROLL_ID,
        excluded_archetypes=(TOP_DECK_ARCHETYPE,),
    )
    selected_archetype_summary = archetype_isolation_candidates[
        archetype_isolation_candidates['deck_archetype'].isin(archetype_roll.selected)
    ].copy()
    selection_order = {
        archetype: index for index, archetype in enumerate(archetype_roll.selected)
    }
    selected_archetype_summary['selection_order'] = (
        selected_archetype_summary['deck_archetype'].map(selection_order)
    )
    selected_archetype_summary = selected_archetype_summary.sort_values('selection_order')
    selected_archetype_summary = selected_archetype_summary[
        [
            'deck_archetype', 'core_card_names', 'replays', 'uses',
            'win_rate', 'unique_exact_decks',
            'core_card_other_archetype_decks',
            'core_card_other_archetype_replays',
        ]
    ]
    selected_archetype_decks = build_archetype_selection_details(
        annotated_facts, archetype_roll.selected
    )
    print('Selected archetypes:')
    display(selected_archetype_summary)
    print('Selected archetype exact decks:')
    display(selected_archetype_decks)
    print('Selected unique replays:', archetype_roll.replay_count)
    print('Sampling attempts:', archetype_roll.attempts)
    selected_archetype_decks.to_csv(
        ARCHETYPE_SELECTION_PATH, index=False, encoding='utf-8-sig'
    )
    print('Saved:', ARCHETYPE_SELECTION_PATH)
except (IsolationSamplingError, ValueError) as exc:
    print('Archetype Isolation roll failed:', exc)
    if isinstance(exc, IsolationSamplingError):
        display(pd.DataFrame([exc.diagnostics]))
    print('Existing selection CSV was not overwritten.')

## Combined Audit

This audit simulates removing all three selected replay unions together. It checks exact-deck, Card-ID, same-archetype, selected-archetype, and exact-top-deck exclusion conditions without forbidding the top deck as an opponent.

In [ ]:
if deck_roll is None or archetype_roll is None or top_deck_roll is None:
    print('Combined audit unavailable: all three roll cells must succeed.')
else:
    combined_selected_deck_ids = (
        tuple(deck_roll.selected) + tuple(top_deck_roll.selected)
    )
    combined_audit = audit_deck_archetype_selection(
        annotated_facts,
        selected_deck_ids=combined_selected_deck_ids,
        selected_archetypes=archetype_roll.selected,
    )
    ordinary_replays = set(
        annotated_facts.loc[
            annotated_facts['deck_id'].isin(deck_roll.selected), 'replay_key'
        ].astype(str)
    )
    top_variant_replays = set(
        annotated_facts.loc[
            annotated_facts['deck_id'].isin(top_deck_roll.selected), 'replay_key'
        ].astype(str)
    )
    combined_audit['ordinary_deck_isolation_replays'] = len(ordinary_replays)
    combined_audit['top_deck_archetype_isolation_replays'] = len(top_variant_replays)
    combined_audit['ordinary_top_variant_overlap_replays'] = len(
        ordinary_replays & top_variant_replays
    )
    combined_audit['top_deck_selected'] = (
        TOP_DECK_ID in combined_selected_deck_ids
    )
    combined_audit['valid'] = (
        combined_audit['valid'] and not combined_audit['top_deck_selected']
    )
    display(pd.DataFrame([combined_audit]))

## Next Steps

Review all three selected tables and the combined audit. Change the corresponding `ROLL_ID` and rerun its cell until the selection is acceptable. The three CSV files under `imitation_learning/data/` are then ready for the later training-data split task.